# GRU Prefetcher V9 -- delta-bitmap + in-distribution

**What V8 taught us** (paste your V8 summary if you want to compare):
- V8 had test top-1 = 47.9% (123x above chance) but IPC = 0.9604x (-4%)
- The accuracy *did* improve over V1 (1.3% top-1) but IPC did not.
- Why: omnetpp's IPC is insensitive to prefetching in a 25M window
  (slide 8: LRU 0.2806, idealized SPP 0.2833 -- spread is only 1%).
- Cross-binary (mcf -> omnetpp) is the wrong evaluation protocol.
  Voyager, Hashemi 2018, and Pythia all train per-application.

**V9 changes** (each one independent, can ablate):

| # | Change | Source |
|---|--------|--------|
| A | **In-distribution split** (train[0:70%], val[70:85%], test[85:100%] on same trace) | Voyager ASPLOS'21, Pythia MICRO'21, Hashemi ICML'18 |
| B | **Delta-bitmap output** (predict which of K nearby deltas in next-8 window) | TransFetch CF'22, DART arXiv 2401.06362 |
| C | **PC+Delta hash feature** (Pythia's winning state vector) | Pythia MICRO'21, Section 4 |
| D | **Sweep K traces** (lbm + gcc + mcf in-distribution) | slide 8 sensitivity ranking |

**Expected results**:
- top-1 accuracy: 40-70% on lbm, 30-50% on mcf in-distribution
- F1 (bitmap): >0.5
- IPC vs baseline: positive on lbm (it has +18% headroom), close to baseline on mcf

This notebook produces ONE prefetch list per trace it trains on.
Run `scripts/dump_trace.sh` for each trace first.


In [58]:
# install optional wandb
import subprocess, sys
try: import wandb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'], check=False)


In [59]:
import os, time, math, json, random, collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print('device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

USE_WANDB = False
try:
    import wandb
    if os.environ.get('WANDB_API_KEY'):
        USE_WANDB = True
        print('wandb: ENABLED')
    else:
        print('wandb: disabled (set WANDB_API_KEY to enable)')
except ImportError:
    print('wandb: not installed')


device: cuda
GPU: Tesla T4
wandb: disabled (set WANDB_API_KEY to enable)


## 1. Configuration

**Pick a trace.** Each run trains/tests on ONE trace (in-distribution).
Recommended order:
  1. `605.mcf_s-994B`  -- direct comparison vs V1..V8
  2. `619.lbm_s-4268B` -- streaming, big prefetch headroom
  3. `602.gcc_s-734B`  -- branch-heavy, biggest prefetch headroom (+139% with SPP)


In [60]:
# --- Trace (in-distribution: train+test on this single trace) ---
TRACE_CSV = '/content/access_trace.602.gcc_s-734B.csv'   # change me

# --- Delta-bitmap output (TransFetch CF'22 / DART arXiv 2401.06362) ---
# Predict which of D nearby cache-line deltas appear in the NEXT_WINDOW future accesses.
DELTA_RANGE   = 64    # bitmap covers deltas [-DELTA_RANGE, +DELTA_RANGE] = 129 bits
NEXT_WINDOW   = 8     # set label[d] = 1 if delta d appears within next NEXT_WINDOW accesses

# --- Input features ---
HIST          = 16    # past delta-token history length
DELTA_VOCAB_K = 2048  # top-K most frequent deltas in input vocabulary (>>256 in V8)
NUM_PC_BUCKETS = 1024 # PC hash bucket count
NUM_PCDELTA_BUCKETS = 4096   # PC+Delta hash bucket count (Pythia winning state)

# --- Architecture ---
HIDDEN      = 96
EMB_DELTA   = 32
EMB_PC      = 24
EMB_PCDELTA = 32
DROPOUT     = 0.1

# --- Training ---
EPOCHS      = 8
BATCH       = 1024
LR          = 1e-3
LR_MIN      = 1e-5
WEIGHT_DECAY= 1e-5
GRAD_CLIP   = 1.0

# --- Data ---
USE_L1DM_ONLY = True
LINE_BITS     = 6
PAGE_BITS     = 12

# --- Inference / export ---
PROB_THRESHOLD     = 0.30   # bitmap probability threshold for emitting a prefetch
MAX_PREFETCH_DEGREE = 2     # at most 2 prefetches per access (TransFetch reports degree 1-4 is best)

VOCAB_SIZE  = DELTA_VOCAB_K + 1   # +1 for OOV
BITMAP_SIZE = 2 * DELTA_RANGE + 1 # = 129

TRACE_TAG = TRACE_CSV.split('.')[1] if '.' in TRACE_CSV else TRACE_CSV
RUN_NAME  = f'V9_{TRACE_TAG}_h{HIDDEN}_hist{HIST}_v{DELTA_VOCAB_K}_bm{BITMAP_SIZE}_w{NEXT_WINDOW}'
print(f'config: {RUN_NAME}')

if USE_WANDB:
    wandb.init(project='gru-prefetcher-v9', name=RUN_NAME,
        config={'trace': TRACE_CSV, 'hist': HIST, 'vocab': VOCAB_SIZE,
                'hidden': HIDDEN, 'bitmap_size': BITMAP_SIZE,
                'next_window': NEXT_WINDOW, 'prob_threshold': PROB_THRESHOLD,
                'epochs': EPOCHS, 'batch': BATCH, 'lr': LR,
                'l1dm_only': USE_L1DM_ONLY})


config: V9_602_h96_hist16_v2048_bm129_w8


## 2. Load + optionally filter to L1 demand misses

In [61]:
raw = pd.read_csv(TRACE_CSV)

print(raw.columns)
print("rows =", len(raw))
print("NaN pc_hex =", raw["pc_hex"].isna().sum())
print("NaN addr_hex =", raw["addr_hex"].isna().sum())
print("NaN hit =", raw["hit"].isna().sum())

bad = raw[raw["pc_hex"].isna() | raw["addr_hex"].isna() | raw["hit"].isna()]
bad.head(20)

Index(['idx', 'addr_hex', 'pc_hex', 'hit'], dtype='object')
rows = 1740794
NaN pc_hex = 1
NaN addr_hex = 0
NaN hit = 1


,idx,addr_hex,pc_hex,hit
1740793,1740793,0xdf7,NaN,NaN


In [62]:
def load_csv(path, use_l1dm_only):
    t0 = time.time()
    df = pd.read_csv(path)

    # Drop malformed rows before int conversion
    need_cols = ['addr_hex', 'pc_hex', 'hit']
    bad = df[df[need_cols].isna().any(axis=1)]
    if len(bad) > 0:
        print(f'[warn] dropping {len(bad):,} malformed rows with NaN in {need_cols}')
        print(bad.head(5))
        df = df.dropna(subset=need_cols).reset_index(drop=True)

    def parse_int(x):
        if isinstance(x, str):
            x = x.strip()
            if x.startswith('0x') or x.startswith('0X'):
                return int(x, 16)
            return int(x)
        return int(x)

    df['addr'] = df['addr_hex'].apply(parse_int).astype('int64')
    df['pc'] = df['pc_hex'].apply(parse_int).astype('int64')
    df['hit'] = df['hit'].astype('int8')

    if 'idx' in df.columns:
        df['idx_orig'] = df['idx'].astype('int64')
    else:
        df['idx_orig'] = np.arange(len(df), dtype='int64')

    n_total = len(df)
    if use_l1dm_only:
        df = df[df['hit'] == 0].reset_index(drop=True)

    n_kept = len(df)
    print(f'[load] {n_total:,} -> {n_kept:,} rows ({n_kept/max(1,n_total)*100:.1f}% kept) '
          f'({time.time()-t0:.1f}s)')
    print(f'       unique PCs={df.pc.nunique()}, unique pages={(df.addr//(1<<PAGE_BITS)).nunique()}')
    return df

df = load_csv(TRACE_CSV, USE_L1DM_ONLY)

[warn] dropping 1 malformed rows with NaN in ['addr_hex', 'pc_hex', 'hit']
             idx addr_hex pc_hex  hit
1740793  1740793    0xdf7    NaN  NaN
[load] 1,740,793 -> 408,969 rows (23.5% kept) (3.5s)
       unique PCs=119, unique pages=5887


## 3. In-distribution time-split: train[0:70%], val[70:85%], test[85:100%]

This is the protocol Voyager ASPLOS'21 and Hashemi ICML'18 use. Same workload,
just different time slices. Cross-binary (mcf -> omnetpp) was a mistake.


In [63]:
N = len(df)
i_train_end = int(N * 0.70)
i_val_end   = int(N * 0.85)
print(f'[split] train [0:{i_train_end:,}]')
print(f'[split] val   [{i_train_end:,}:{i_val_end:,}]')
print(f'[split] test  [{i_val_end:,}:{N:,}]')


[split] train [0:286,278]
[split] val   [286,278:347,623]
[split] test  [347,623:408,969]


## 4. Build delta vocabulary from train portion only

In [64]:
def build_delta_vocab(df, end_idx, K):
    addrs = df['addr'].values[:end_idx]
    pcs   = df['pc'].values[:end_idx]
    last = {}
    cnt = collections.Counter()
    for i in range(end_idx):
        pc, a = int(pcs[i]), int(addrs[i])
        if pc in last:
            d = (a - last[pc]) >> LINE_BITS
            cnt[d] += 1
        last[pc] = a
    top = [d for d, _ in cnt.most_common(K)]
    delta_to_id = {d: i+1 for i, d in enumerate(top)}     # 1..K (0=OOV)
    id_to_delta = {i+1: d for i, d in enumerate(top)}
    total = sum(cnt.values())
    covered = sum(cnt[d] for d in top)
    coverage = covered / max(1, total)
    print(f'[vocab] built from {total:,} train deltas, '
          f'top-{K} coverage = {coverage*100:.1f}% '
          f'({covered:,}/{total:,} occurrences)')
    print(f'[vocab] top-10 deltas (cache lines): {top[:10]}')
    print(f'[vocab] head deltas distribution: {[f"{d}:{cnt[d]:,}" for d in top[:5]]}')
    return delta_to_id, id_to_delta, coverage

delta_to_id, id_to_delta, vocab_coverage = build_delta_vocab(df, i_train_end, DELTA_VOCAB_K)
if USE_WANDB:
    wandb.log({'vocab/coverage_train': vocab_coverage})


[vocab] built from 286,161 train deltas, top-2048 coverage = 90.2% (258,182/286,161 occurrences)
[vocab] top-10 deltas (cache lines): [1, 3, 0, 5, 2, 6, -1, 10, 8, 15]
[vocab] head deltas distribution: ['1:178,939', '3:34,784', '0:31,816', '5:1,537', '2:1,476']


## 5. Featurize (input + delta-bitmap label)

For each access i:
- **input**: last 16 delta-vocab tokens (per-PC) + PC bucket + PC+Δ hash
- **label**: bitmap[k] = 1 iff (one of accesses i+1..i+NEXT_WINDOW has delta = k - DELTA_RANGE)
            where k indexes positions [-DELTA_RANGE, +DELTA_RANGE]

This is set prediction, not sequence prediction. Lets the model predict
variable-degree prefetches naturally.


In [65]:
def featurize_v9(df, delta_to_id, hist_len, delta_range, next_window):
    addrs = df['addr'].values
    pcs   = df['pc'].values
    idxs  = df['idx_orig'].values
    N = len(df)

    # Per-PC running delta-token history
    last_addrs = {}
    hist_ids   = {}

    X_delta   = np.zeros((N, hist_len), dtype=np.int64)
    X_pc      = np.zeros(N, dtype=np.int64)
    X_pcdelta = np.zeros(N, dtype=np.int64)
    Y_bitmap  = np.zeros((N, 2*delta_range+1), dtype=np.float32)
    Idx       = np.zeros(N, dtype=np.int64)
    Cur       = np.zeros(N, dtype=np.int64)
    keep      = np.zeros(N, dtype=bool)

    OOV = 0
    BITMAP_SIZE = 2*delta_range + 1
    CENTER = delta_range

    # Pre-compute all per-access deltas (memory > recomputation)
    last_for_delta = {}
    access_deltas = np.zeros(N, dtype=np.int64)
    for i in range(N):
        pc, a = int(pcs[i]), int(addrs[i])
        if pc in last_for_delta:
            d = (a - last_for_delta[pc]) >> LINE_BITS
        else:
            d = 0
        access_deltas[i] = d
        last_for_delta[pc] = a

    # Build per-PC histories and labels
    for i in range(N - 1):
        pc, a = int(pcs[i]), int(addrs[i])

        # Update history: most recent delta of this PC
        if pc in last_addrs:
            d_prev = (a - last_addrs[pc]) >> LINE_BITS
            tok = delta_to_id.get(d_prev, OOV)
            h = hist_ids.setdefault(pc, [])
            h.append(tok)
            if len(h) > hist_len:
                hist_ids[pc] = h[-hist_len:]
        last_addrs[pc] = a

        h = hist_ids.get(pc, [])
        if len(h) < 1: continue

        # Pad to hist_len (oldest first; most recent at index hist_len-1)
        padded = [OOV] * (hist_len - len(h)) + h[-hist_len:]
        X_delta[i] = padded
        X_pc[i]    = pc % NUM_PC_BUCKETS

        # Pythia's "PC+Delta" feature: hash(PC ^ most_recent_delta)
        last_d = h[-1] if h else 0
        pcdelta_hash = (pc ^ (last_d * 2654435761)) & (NUM_PCDELTA_BUCKETS - 1)
        X_pcdelta[i] = pcdelta_hash

        # ---- Label = delta-bitmap over next NEXT_WINDOW accesses ----
        # NOTE: the deltas we put in the bitmap are deltas FROM CURRENT addr a
        # to each of the next-window accesses (not per-PC deltas).
        cur_line = a >> LINE_BITS
        for j in range(1, next_window + 1):
            if i + j >= N: break
            next_addr = int(addrs[i + j])
            d_line = (next_addr >> LINE_BITS) - cur_line
            if -delta_range <= d_line <= delta_range:
                Y_bitmap[i, CENTER + d_line] = 1.0

        Idx[i] = idxs[i]
        Cur[i] = a
        keep[i] = True

    return (X_delta[keep], X_pc[keep], X_pcdelta[keep],
            Y_bitmap[keep], Idx[keep], Cur[keep])

print('[featurize] computing...')
X_d, X_pc, X_pcd, Y_bm, Idx, Cur = featurize_v9(
    df, delta_to_id, HIST, DELTA_RANGE, NEXT_WINDOW)
print(f'  -> {X_d.shape[0]:,} examples, bitmap size {Y_bm.shape[1]}')

# Stats on bitmap labels
density = Y_bm.sum(axis=1)
print(f'[stat] avg #set bits per label = {density.mean():.2f}  (max possible {NEXT_WINDOW})')
print(f'[stat] fraction of labels with zero set bits = {(density==0).mean()*100:.1f}%')

# Build splits using the time indices we already established
# Recompute end positions on the FILTERED arrays (kept[] is monotonic in index)
# Both i_train_end and i_val_end refer to positions in the original df.
# After keep[]-filtering, we need to map them.
# Simple approach: split kept arrays proportionally.
nkept = X_d.shape[0]
i_tr = int(0.70 * nkept); i_va = int(0.85 * nkept)
print(f'[split] (kept) train [0:{i_tr:,}]  val [{i_tr:,}:{i_va:,}]  test [{i_va:,}:{nkept:,}]')


[featurize] computing...
  -> 408,849 examples, bitmap size 129
[stat] avg #set bits per label = 4.73  (max possible 8)
[stat] fraction of labels with zero set bits = 5.2%
[split] (kept) train [0:286,194]  val [286,194:347,521]  test [347,521:408,849]


## 6. DataLoader

In [66]:
class PFDS(Dataset):
    def __init__(self, X_d, X_pc, X_pcd, Y_bm):
        self.X_d   = torch.from_numpy(X_d)
        self.X_pc  = torch.from_numpy(X_pc)
        self.X_pcd = torch.from_numpy(X_pcd)
        self.Y     = torch.from_numpy(Y_bm)
    def __len__(self): return len(self.Y)
    def __getitem__(self, i):
        return self.X_d[i], self.X_pc[i], self.X_pcd[i], self.Y[i]

tr_ds = PFDS(X_d[:i_tr],  X_pc[:i_tr],  X_pcd[:i_tr],  Y_bm[:i_tr])
va_ds = PFDS(X_d[i_tr:i_va], X_pc[i_tr:i_va], X_pcd[i_tr:i_va], Y_bm[i_tr:i_va])
te_ds = PFDS(X_d[i_va:], X_pc[i_va:], X_pcd[i_va:], Y_bm[i_va:])

tr_ld = DataLoader(tr_ds, batch_size=BATCH, shuffle=True,  drop_last=True)
va_ld = DataLoader(va_ds, batch_size=BATCH, shuffle=False, drop_last=False)
te_ld = DataLoader(te_ds, batch_size=BATCH, shuffle=False, drop_last=False)


## 7. GRU + delta-bitmap head

Architecture:
- delta-token embedding -> GRU(hidden=96)
- concat[h_last, pc_emb, pcdelta_emb]
- 2-layer MLP -> bitmap_size logits (sigmoid for BCE)

This is the same recurrent core as d2l.ai chapter 10.2 GRU, but the head
is a multi-label classifier (BCE), not a single-class softmax.

The PC+Delta hash embedding is Pythia's winning state-vector idea (MICRO'21):
the most predictive feature is not PC or delta alone but their joint hash.


In [67]:
class V9Prefetcher(nn.Module):
    def __init__(self, vocab_size, num_pc, num_pcd, hidden,
                 emb_d, emb_pc, emb_pcd, bitmap_size, dropout):
        super().__init__()
        self.embed_d   = nn.Embedding(vocab_size, emb_d, padding_idx=0)
        self.embed_pc  = nn.Embedding(num_pc, emb_pc)
        self.embed_pcd = nn.Embedding(num_pcd, emb_pcd)
        self.gru = nn.GRU(emb_d, hidden, batch_first=True)
        side = hidden + emb_pc + emb_pcd
        self.head = nn.Sequential(
            nn.Linear(side, side), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(side, bitmap_size),
        )
    def forward(self, d_hist, pc, pcd):
        d_emb = self.embed_d(d_hist)
        _, h = self.gru(d_emb)
        h_last = h[-1]
        pc_emb  = self.embed_pc(pc)
        pcd_emb = self.embed_pcd(pcd)
        z = torch.cat([h_last, pc_emb, pcd_emb], dim=1)
        return self.head(z)        # logits

model = V9Prefetcher(VOCAB_SIZE, NUM_PC_BUCKETS, NUM_PCDELTA_BUCKETS,
                     HIDDEN, EMB_DELTA, EMB_PC, EMB_PCDELTA,
                     BITMAP_SIZE, DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[model] {n_params:,} params')
if USE_WANDB:
    wandb.log({'model/params': n_params})


[model] 301,649 params


## 8. Training loop

Loss: BCE over the bitmap, weighted (label imbalance: most bits are 0).
Metrics each epoch:
- BCE loss
- top-1 accuracy: did argmax of logits hit a set bit in the label?
- F1@0.3: standard F1 at the deployment threshold


In [68]:
def evaluate(model, loader, device, thr=PROB_THRESHOLD):
    model.eval()
    n = 0
    top1 = 0; topk = {1:0, 3:0, 5:0}
    tp = fp = fn = 0
    loss_sum = 0.0
    with torch.no_grad():
        for X_d, X_pc, X_pcd, Y in loader:
            X_d, X_pc, X_pcd, Y = X_d.to(device), X_pc.to(device), X_pcd.to(device), Y.to(device)
            logits = model(X_d, X_pc, X_pcd)
            loss = F.binary_cross_entropy_with_logits(logits, Y, reduction='sum')
            loss_sum += loss.item()
            probs = torch.sigmoid(logits)
            # top-K accuracy: argmax (or top-k) is in the set of true positions
            for k in (1, 3, 5):
                tk = probs.topk(k, dim=1).indices                 # [B, k]
                hit = Y.gather(1, tk).max(dim=1).values > 0.5
                topk[k] += hit.sum().item()
            top1 = topk[1]
            # F1 at threshold
            pred = (probs >= thr).float()
            tp += (pred * Y).sum().item()
            fp += (pred * (1 - Y)).sum().item()
            fn += ((1 - pred) * Y).sum().item()
            n += Y.size(0)
    precision = tp / max(1, tp + fp)
    recall    = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-9, precision + recall)
    return {
        'loss': loss_sum / max(1, n * BITMAP_SIZE),
        'top1': topk[1] / max(1, n),
        'top3': topk[3] / max(1, n),
        'top5': topk[5] / max(1, n),
        'precision': precision, 'recall': recall, 'f1': f1,
        'n': n,
    }

# Positive class weight: bitmaps are sparse, so we upweight positives
# We don't recompute pos_weight per batch -- just use a fixed estimate from train labels
pos_per_label = Y_bm[:i_tr].mean(axis=0).clip(1e-6, None).mean()  # rough avg positive rate
pos_weight = torch.tensor([(1 - pos_per_label) / pos_per_label]).to(DEVICE)
print(f'[loss] avg label positive rate = {pos_per_label:.4f}, pos_weight = {pos_weight.item():.1f}')

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt, T_max=EPOCHS * len(tr_ld), eta_min=LR_MIN)

history = []
best_val_f1 = -1.0
t0 = time.time()

for ep in range(EPOCHS):
    model.train()
    running = 0.0; nb = 0; ok = 0; tot = 0
    ep_t0 = time.time()
    for X_d_b, X_pc_b, X_pcd_b, Y in tr_ld:
        X_d_b = X_d_b.to(DEVICE); X_pc_b = X_pc_b.to(DEVICE)
        X_pcd_b = X_pcd_b.to(DEVICE); Y = Y.to(DEVICE)
        logits = model(X_d_b, X_pc_b, X_pcd_b)
        loss = F.binary_cross_entropy_with_logits(logits, Y, pos_weight=pos_weight)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step(); sched.step()
        running += loss.item(); nb += 1
        # train-time top-1 (just argmax hits a positive)
        with torch.no_grad():
            pred1 = torch.sigmoid(logits).argmax(dim=1)
            hit = Y.gather(1, pred1.unsqueeze(1)).squeeze(1) > 0.5
            ok += hit.sum().item(); tot += Y.size(0)
    train_loss = running / max(1, nb)
    train_top1 = ok / max(1, tot)
    val_m = evaluate(model, va_ld, DEVICE)
    elapsed = time.time() - ep_t0
    history.append({'epoch': ep+1, 'train_loss': train_loss, 'train_top1': train_top1,
                    **{f'val_{k}': v for k, v in val_m.items()},
                    'lr': sched.get_last_lr()[0], 'epoch_s': elapsed})
    print(f'  ep {ep+1}/{EPOCHS}  loss={train_loss:.4f}  '
          f'train_top1={train_top1:.3f}  val_top1={val_m["top1"]:.3f}  '
          f'val_top5={val_m["top5"]:.3f}  val_f1={val_m["f1"]:.3f}  '
          f'(P={val_m["precision"]:.2f}/R={val_m["recall"]:.2f})  '
          f'lr={sched.get_last_lr()[0]:.2e}  ({elapsed:.1f}s)')
    if USE_WANDB:
        wandb.log({'epoch': ep+1, 'train/loss': train_loss, 'train/top1': train_top1,
                   'val/top1': val_m['top1'], 'val/top5': val_m['top5'],
                   'val/f1': val_m['f1'], 'val/precision': val_m['precision'],
                   'val/recall': val_m['recall'], 'lr': sched.get_last_lr()[0]})
    if val_m['f1'] > best_val_f1:
        best_val_f1 = val_m['f1']

print(f'\n[train] total time: {time.time()-t0:.1f}s')
print(f'[train] best val F1: {best_val_f1:.4f}')

test_m = evaluate(model, te_ld, DEVICE)
print(f'\n[test, in-distribution, last 15% of {TRACE_CSV}]')
print(f'  top-1: {test_m["top1"]:.4f}')
print(f'  top-3: {test_m["top3"]:.4f}')
print(f'  top-5: {test_m["top5"]:.4f}')
print(f'  F1@{PROB_THRESHOLD}: {test_m["f1"]:.4f}  (P={test_m["precision"]:.3f}, R={test_m["recall"]:.3f})')
if USE_WANDB:
    wandb.log({'test/top1': test_m['top1'], 'test/top3': test_m['top3'],
               'test/top5': test_m['top5'], 'test/f1': test_m['f1']})


[loss] avg label positive rate = 0.0425, pos_weight = 22.5
  ep 1/8  loss=0.1349  train_top1=0.894  val_top1=0.944  val_top5=0.967  val_f1=0.786  (P=0.65/R=1.00)  lr=9.62e-04  (7.0s)
  ep 2/8  loss=0.0654  train_top1=0.915  val_top1=0.949  val_top5=0.968  val_f1=0.815  (P=0.69/R=1.00)  lr=8.55e-04  (7.2s)
  ep 3/8  loss=0.0623  train_top1=0.917  val_top1=0.949  val_top5=0.968  val_f1=0.819  (P=0.69/R=1.00)  lr=6.94e-04  (6.4s)
  ep 4/8  loss=0.0609  train_top1=0.918  val_top1=0.949  val_top5=0.969  val_f1=0.842  (P=0.73/R=1.00)  lr=5.05e-04  (7.2s)
  ep 5/8  loss=0.0602  train_top1=0.919  val_top1=0.950  val_top5=0.968  val_f1=0.824  (P=0.70/R=1.00)  lr=3.16e-04  (6.4s)
  ep 6/8  loss=0.0597  train_top1=0.919  val_top1=0.950  val_top5=0.968  val_f1=0.823  (P=0.70/R=1.00)  lr=1.55e-04  (7.2s)
  ep 7/8  loss=0.0595  train_top1=0.919  val_top1=0.950  val_top5=0.969  val_f1=0.825  (P=0.70/R=1.00)  lr=4.77e-05  (6.2s)
  ep 8/8  loss=0.0594  train_top1=0.919  val_top1=0.950  val_top5=0.968  

## 9. CPU latency benchmark

In [69]:
def bench_cpu_us(m, hist, n=1000):
    m_cpu = m.to('cpu').eval()
    d = torch.zeros((1, hist), dtype=torch.long)
    p = torch.zeros((1,), dtype=torch.long)
    pd_ = torch.zeros((1,), dtype=torch.long)
    with torch.no_grad():
        for _ in range(20): m_cpu(d, p, pd_)
        t = time.perf_counter()
        for _ in range(n): m_cpu(d, p, pd_)
        us = (time.perf_counter() - t) / n * 1e6
    m.to(DEVICE)
    return us

inf_us = bench_cpu_us(model, HIST)
print(f'[bench] CPU inference: {inf_us:.1f} us / call')
if USE_WANDB:
    wandb.log({'bench/cpu_us': inf_us})


[bench] CPU inference: 830.7 us / call


## 10. Export prefetch list

For each access in the test split:
- score = sigmoid(logits) over the 129-position bitmap
- top-K positions with prob >= PROB_THRESHOLD emit prefetches (K = MAX_PREFETCH_DEGREE)
- each prefetch target = current_addr + (delta_position - CENTER) * 64 bytes

This naturally supports variable prefetch degree (0-K depending on confidence),
the way TransFetch CF'22 does. If a model is sure about 2 future cache lines,
issue 2 prefetches; if sure about none, issue 0.


In [70]:
out_path = f'prefetch_list_GRU_V9_{TRACE_TAG}.txt'
n_total = 0; n_emit = 0
all_max_probs = []

CENTER = DELTA_RANGE
BS = 4096
model.to(DEVICE).eval()

with open(out_path, 'w') as fh, torch.no_grad():
    # We score ONLY the test slice (same ChampSim window the IPC will be measured on)
    Xd_te  = X_d[i_va:];   Xpc_te = X_pc[i_va:]
    Xpcd_te = X_pcd[i_va:]; Idx_te = Idx[i_va:];  Cur_te = Cur[i_va:]
    for i in range(0, len(Idx_te), BS):
        d = torch.from_numpy(Xd_te[i:i+BS]).to(DEVICE)
        p = torch.from_numpy(Xpc_te[i:i+BS]).to(DEVICE)
        pd_ = torch.from_numpy(Xpcd_te[i:i+BS]).to(DEVICE)
        logits = model(d, p, pd_)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_max_probs.append(probs.max(axis=1))
        idxs = Idx_te[i:i+BS]
        curs = Cur_te[i:i+BS]
        for j in range(probs.shape[0]):
            n_total += 1
            row = probs[j]
            # top-K above threshold
            top_idx = np.argsort(-row)[:MAX_PREFETCH_DEGREE]
            emitted_this = 0
            for ti in top_idx:
                if row[ti] < PROB_THRESHOLD: break
                d_lines = int(ti) - CENTER
                if d_lines == 0: continue
                pf_addr = (int(curs[j]) + (d_lines << LINE_BITS)) & 0xffffffffffffffff
                if pf_addr <= 0: continue
                fh.write(f'{int(idxs[j])} 0x{pf_addr:x}\n')
                n_emit += 1
                emitted_this += 1

all_max_probs = np.concatenate(all_max_probs) if all_max_probs else np.array([])
trigger_rate_access = n_total > 0 and (n_emit > 0)   # not quite right; we want unique idx
unique_idx_count = n_total  # number of accesses we scored
trigger_pct = 100.0 * n_emit / max(1, unique_idx_count * MAX_PREFETCH_DEGREE)
print(f'[export] wrote {out_path}')
print(f'  accesses scored : {unique_idx_count:,}')
print(f'  prefetches emitted: {n_emit:,}  '
      f'({n_emit / max(1, unique_idx_count) * 100:.1f}% access trigger rate, '
      f'avg degree {n_emit / max(1, unique_idx_count):.2f})')
print(f'\n[confidence] top-1 sigmoid percentiles:')
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'  p{p:>2d} = {np.percentile(all_max_probs, p):.3f}')


[export] wrote prefetch_list_GRU_V9_602.txt
  accesses scored : 61,328
  prefetches emitted: 82,396  (134.4% access trigger rate, avg degree 1.34)

[confidence] top-1 sigmoid percentiles:
  p10 = 0.997
  p25 = 0.999
  p50 = 0.999
  p75 = 0.999
  p90 = 1.000
  p95 = 1.000
  p99 = 1.000


## 11. JSON summary

In [71]:
summary = {
    'run_name': RUN_NAME,
    'trace': TRACE_CSV,
    'config': {
        'hist': HIST, 'vocab_size': VOCAB_SIZE,
        'bitmap_size': BITMAP_SIZE, 'delta_range': DELTA_RANGE,
        'next_window': NEXT_WINDOW,
        'hidden': HIDDEN, 'num_pc_buckets': NUM_PC_BUCKETS,
        'num_pcdelta_buckets': NUM_PCDELTA_BUCKETS,
        'epochs': EPOCHS, 'batch': BATCH, 'lr': LR,
        'l1dm_only': USE_L1DM_ONLY,
        'prob_threshold': PROB_THRESHOLD,
        'max_prefetch_degree': MAX_PREFETCH_DEGREE,
    },
    'split': {
        'i_train_end_kept': int(i_tr),
        'i_val_end_kept':   int(i_va),
        'n_kept_total':     int(X_d.shape[0]),
    },
    'data_stats': {
        'vocab_coverage_train': float(vocab_coverage),
        'avg_label_density': float(density.mean()),
        'zero_label_frac':   float((density == 0).mean()),
    },
    'metrics': {
        'best_val_f1': float(best_val_f1),
        'test_top1': float(test_m['top1']),
        'test_top3': float(test_m['top3']),
        'test_top5': float(test_m['top5']),
        'test_f1':   float(test_m['f1']),
        'test_precision': float(test_m['precision']),
        'test_recall':    float(test_m['recall']),
        'cpu_inf_us': float(inf_us),
    },
    'prefetch_list': {
        'path': out_path,
        'n_emitted': int(n_emit),
        'n_total_accesses': int(unique_idx_count),
        'trigger_per_access': float(n_emit / max(1, unique_idx_count)),
    },
    'history': history,
    'params': int(n_params),
}

with open(f'gru_v9_summary_{TRACE_TAG}.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
print(f'saved gru_v9_summary_{TRACE_TAG}.json')
print()
print('===== HEADLINE =====')
print(f'  trace         : {TRACE_CSV}  (in-distribution!)')
print(f'  test top-1    : {test_m["top1"]:.4f}')
print(f'  test top-5    : {test_m["top5"]:.4f}')
print(f'  test F1@{PROB_THRESHOLD} : {test_m["f1"]:.4f}')
print(f'  prefetch list : {out_path}  ({n_emit:,} prefetches)')
print(f'  CPU inference : {inf_us:.0f} us')
print()
print('Next steps on lab machine:')
print(f'  scp this:prefetch_list_GRU_V9_{TRACE_TAG}.txt lab:$WORKDIR/')
print(f'  TRACE={TRACE_TAG} \\')
print(f'      PFETCH=$WORKDIR/prefetch_list_GRU_V9_{TRACE_TAG}.txt \\')
print(f'      MODEL_TAG=GRU_V9_{TRACE_TAG} \\')
print(f'      WARMUP=21250000 SIM=3750000 \\')
print(f'      bash scripts/run_nn_replay.sh')
print()
print('IMPORTANT: because we trained on first 70% and exported prefetches for the')
print('LAST 15% of the trace, the ChampSim run must seek to the same window.')
print('Easiest: warmup-instructions = (instructions covered by first 85%),')
print('         simulation-instructions = (instructions covered by last 15%).')
print('Since trace dumper sees only L1DM accesses and ChampSim sees every instr,')
print('these numbers depend on the trace. For mcf at 25M dumped: warmup=21.25M, sim=3.75M.')

if USE_WANDB:
    wandb.finish()


saved gru_v9_summary_602.json

===== HEADLINE =====
  trace         : /content/access_trace.602.gcc_s-734B.csv  (in-distribution!)
  test top-1    : 0.9476
  test top-5    : 0.9655
  test F1@0.3 : 0.8346
  prefetch list : prefetch_list_GRU_V9_602.txt  (82,396 prefetches)
  CPU inference : 831 us

Next steps on lab machine:
  scp this:prefetch_list_GRU_V9_602.txt lab:$WORKDIR/
  TRACE=602 \
      PFETCH=$WORKDIR/prefetch_list_GRU_V9_602.txt \
      MODEL_TAG=GRU_V9_602 \
      WARMUP=21250000 SIM=3750000 \
      bash scripts/run_nn_replay.sh

IMPORTANT: because we trained on first 70% and exported prefetches for the
LAST 15% of the trace, the ChampSim run must seek to the same window.
Easiest: warmup-instructions = (instructions covered by first 85%),
         simulation-instructions = (instructions covered by last 15%).
Since trace dumper sees only L1DM accesses and ChampSim sees every instr,
these numbers depend on the trace. For mcf at 25M dumped: warmup=21.25M, sim=3.75M.
